# REBULEX — farklı embedding modelleriyle ölçüm (Colab)

Bu notebook seçilen embedding modeliyle 10 binlik karar havuzunda temsilci parçalama denemelerini kurar ve
**v2 sorguları** ile **dilekçeler** (`dilekce_v1`) üzerinde ölçer. Kod GitHub'dan, veriler Drive'dan gelir.

**Başlamadan önce**
1. `Çalışma zamanı > Çalışma zamanı türünü değiştir` → **GPU (L4 ya da A100)**. T4 yavaştır.
2. Drive'da `REBULEX/data/` klasörü olmalı (içinde `n10000/kararlar.parquet`, `test_sets/`, `experiments.csv`).
3. Yalnızca **embeddinggemma** için: Hugging Face'te model sayfasında lisansı kabul et, soldaki 🔑 *Secrets*
   bölümüne `HF_TOKEN` adıyla token ekle ve notebook erişimini aç.

**Akış:** 1–5. hücreler oturum başında bir kez çalışır. Sonra her model için: 2. hücrede `MODEL`'i seç → 2. ve 6–9. hücreler.
Aynı anda yalnızca bir model çalıştır; Colab dönemi boyunca `experiments.csv`'nin asıl kopyası Drive'dakidir.

## 1. Drive'ı bağla ve GPU'yu kontrol et

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Ayarlar ve model seçimi
`MODEL` satırını değiştirerek modeli seç. Değerler `config.py`'nin başındaki yorumla aynıdır.

`REBULEX_BATCH_TOKENS` ve `REBULEX_DTYPE` ekran kartına göre ayarlanır. L4/A100 için aşağıdaki değerler uygundur;
T4 kullanıyorsan `bfloat16` yerine `auto` yaz ve yığını 32768'e düşür.

In [ ]:
import os

DRIVE_DATA = "/content/drive/MyDrive/REBULEX/data"   # Drive'daki data klasörü
LOCAL_DATA = "/content/data"                          # Colab'ın kendi diski (hızlı); denemeler burada çalışır
REPO = "https://github.com/ahmetege0/rebulex.git"
TEST_SETS = "v2 dilekce_v1"

# GPU ayarları: varsayılanlar 6 GB'lık ekran kartına göre; L4/A100'de büyük yığın ve 16 bitlik hesap birkaç kat hızlı
os.environ["REBULEX_BATCH_TOKENS"] = "65536"   # bir yığında işlenecek yaklaşık token (T4 ise 32768 yapın)
os.environ["REBULEX_DTYPE"] = "bfloat16"       # "auto" = modelin kendi ayarı (T4 bfloat16 desteklemez, "auto" bırakın)

MODELS = {
    "magibu_embedding": dict(MODEL_NAME="magibu/embeddingmagibu-200m", MODEL_MAX_TOKENS=8192,
                             QUERY_PROMPT="task: search result | query: ", DOCUMENT_FORMAT="title: {title} | text: {text}"),
    "embeddinggemma":   dict(MODEL_NAME="google/embeddinggemma-300m", MODEL_MAX_TOKENS=2048,
                             QUERY_PROMPT="task: search result | query: ", DOCUMENT_FORMAT="title: {title} | text: {text}"),
    "cosmos_e5":        dict(MODEL_NAME="ytu-ce-cosmos/turkish-e5-large", MODEL_MAX_TOKENS=512,
                             QUERY_PROMPT="Instruct: Given a Turkish search query, retrieve relevant passages written in Turkish that best answer the query\nQuery: ",
                             DOCUMENT_FORMAT="{title}\n{text}"),
    "mursit_large":     dict(MODEL_NAME="newmindai/Mursit-Large-TR-Retrieval", MODEL_MAX_TOKENS=2048,
                             QUERY_PROMPT="", DOCUMENT_FORMAT="{title}\n{text}"),
    "bge_m3":           dict(MODEL_NAME="BAAI/bge-m3", MODEL_MAX_TOKENS=8192,
                             QUERY_PROMPT="", DOCUMENT_FORMAT="{title}\n{text}"),
    "qwen3_06b":        dict(MODEL_NAME="Qwen/Qwen3-Embedding-0.6B", MODEL_MAX_TOKENS=8192,
                             QUERY_PROMPT="Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:",
                             DOCUMENT_FORMAT="{title}\n{text}"),
    "gte_multi":        dict(MODEL_NAME="Alibaba-NLP/gte-multilingual-base", MODEL_MAX_TOKENS=8192,
                             QUERY_PROMPT="", DOCUMENT_FORMAT="{title}\n{text}", TRUST_REMOTE_CODE=True),
}

MODEL = "cosmos_e5"   # <-- çalıştırılacak model

SETTINGS = {"MODEL_FOLDER": MODEL, "TRUST_REMOTE_CODE": False, **MODELS[MODEL]}
print(MODEL, "->", SETTINGS["MODEL_NAME"])

## 3. Kodu GitHub'dan çek ve kur
Depo zaten varsa en son sürüme güncellenir (Colab'daki `config.py` değişikliği silinir; 6. hücre yeniden yazar).

In [ ]:
if not os.path.exists("/content/rebulex"):
    !git clone -q {REPO} /content/rebulex
else:
    !cd /content/rebulex && git fetch -q && git reset -q --hard origin/main
%cd /content/rebulex
!git log --oneline -1
!pip install -q -r requirements.txt
import sentence_transformers, transformers, torch
print("sentence-transformers", sentence_transformers.__version__, "| transformers", transformers.__version__,
      "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())

## 4. Hugging Face token (yalnızca embeddinggemma için gerekli)

In [ ]:
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN yüklendi")
except Exception as e:
    print("HF_TOKEN yok; embeddinggemma dışındaki modeller için sorun değil:", type(e).__name__)

## 5. Gereken verileri Drive'dan Colab diskine kopyala
Sadece kararlar, test setleri ve `experiments.csv` kopyalanır; Drive'daki eski çalışmalara dokunulmaz.
Vektör veritabanı Drive üzerinde değil, Colab'ın diskinde çalışır.

In [ ]:
import shutil

os.makedirs(f"{LOCAL_DATA}/n10000", exist_ok=True)
shutil.copy(f"{DRIVE_DATA}/n10000/kararlar.parquet", f"{LOCAL_DATA}/n10000/kararlar.parquet")
shutil.copytree(f"{DRIVE_DATA}/test_sets", f"{LOCAL_DATA}/test_sets", dirs_exist_ok=True)
if not os.path.exists(f"{LOCAL_DATA}/experiments.csv"):   # oturum içinde eklenen satırlar ezilmesin
    shutil.copy(f"{DRIVE_DATA}/experiments.csv", f"{LOCAL_DATA}/experiments.csv")
os.environ["REBULEX_DATA"] = LOCAL_DATA   # config.py veriyi buradan okur
!ls -la {LOCAL_DATA} {LOCAL_DATA}/n10000 {LOCAL_DATA}/test_sets

## 6. Seçilen modeli `config.py`'ye yaz
`config.py`'deki model satırları 2. hücredeki değerlerle değiştirilir; sonuç aşağıda yazdırılır.

In [ ]:
import re

text = open("config.py", encoding="utf-8").read()
for key, value in SETTINGS.items():
    text, n = re.subn(rf"^{key} = .*$", lambda m: f"{key} = {value!r}", text, flags=re.M)
    assert n == 1, f"config.py'de '{key} = ...' satırı bulunamadı"
open("config.py", "w", encoding="utf-8").write(text)
!grep -n -E "^(DATASET_SIZE|MODEL_NAME|MODEL_FOLDER|MODEL_MAX_TOKENS|QUERY_PROMPT|DOCUMENT_FORMAT|TRUST_REMOTE_CODE) =" config.py

## 7. Denemeleri kur ve ölç
Her deneme için indeks bir kez kurulur, iki test seti de ölçülür. Kesilirse aynı oturumda bu hücre kaldığı yerden
devam eder. Oturum koparsa bu model baştan çalıştırılır.

In [ ]:
!python run_experiments.py --test-sets {TEST_SETS}

## 8. Sonuçları Drive'a kaydet
Modelin klasörü (`chunks`, `embeddings`, `index_info`, `eval`) ve `experiments.csv` Drive'a kopyalanır.
`qdrant` kopyalanmaz; `embeddings` dosyasından yeniden kurulabilir.

In [ ]:
src = f"{LOCAL_DATA}/n10000/{MODEL}"
dst = f"{DRIVE_DATA}/n10000/{MODEL}"
shutil.copytree(src, dst, ignore=shutil.ignore_patterns("qdrant"), dirs_exist_ok=True)
shutil.copy(f"{LOCAL_DATA}/experiments.csv", f"{DRIVE_DATA}/experiments.csv")
for folder in ["chunks", "embeddings", "index_info", "eval"]:
    print(f"{folder:11s}", sorted(os.listdir(f"{dst}/{folder}")))

## 9. Bu modelin sonuçları

In [ ]:
import pandas as pd

exp = pd.read_csv(f"{LOCAL_DATA}/experiments.csv", encoding="utf-8-sig")
cols = ["model", "deneme", "test_seti", "ilk1_%", "ilk5_%", "ilk10_%", "mrr", "ilk5_genis_%", "ilk10_genis_%",
        "ilgili_ilk5_%", "embedding_sn", "olcum_tarihi"]
rows = exp[(exp.model == SETTINGS["MODEL_NAME"]) & (exp.karar_sayisi > 5000)]
rows[[c for c in cols if c in rows]].sort_values(["test_seti", "deneme"])

## Sıradaki model
2. hücrede `MODEL`'i değiştir, 2. hücreyi çalıştır, sonra 6 → 9. hücreleri çalıştır.
Önerilen sıra: `cosmos_e5`, `embeddinggemma`, `mursit_large`, `bge_m3`, `qwen3_06b`, `gte_multi`.